In [10]:

import os
import torch
import timm
from timm.models.vision_transformer import VisionTransformer
from timm.layers import SwiGLUPacked
# --- CONFIGURAZIONE PERCORSI ---
# Se sei su Kaggle usa il path dell'input, o in locale
ON_KAGGLE = os.path.exists("/kaggle")

if ON_KAGGLE:
    PATH_PESI = "/kaggle/input/datasets/irenebartolini/virchow2/uni_backbone_weights.bin"
else:
    PATH_PESI = "./backbone/uni_backbone_weights.bin" 

print("Inizializzazione della struttura di Virchow2")

# Aggiornato mlp_ratio allineandolo alla dimensione combinata di SwiGLUPacked (6832)
backbone = VisionTransformer(
    img_size=224,
    patch_size=14,
    embed_dim=1280,
    depth=32,
    num_heads=16,
    mlp_ratio=6832 / 1280,   # CORREZIONE FINALE: imposta il rapporto a 5.3375
    reg_tokens=4,
    init_values=1e-5,
    mlp_layer=SwiGLUPacked,
    act_layer=torch.nn.SiLU,
    num_classes=0
)

print(f"Caricamento e iniezione dei pesi locali da: {PATH_PESI}")

try:
    stato_modello = torch.load(PATH_PESI, map_location="cpu")
    
    # Pulizia standard se i pesi sono dentro un dizionario nidificato
    if "state_dict" in stato_modello:
        stato_modello = stato_modello["state_dict"]
    elif "model" in stato_modello:
        stato_modello = stato_modello["model"]

    # Carichiamo i pesi
    backbone.load_state_dict(stato_modello)
    backbone.eval()
    print("\n--- [SUCCESSO] Virchow2 è caricato e pronto al 100% in modalità offline! ---")

except Exception as e:
    print(f"\nErrore durante il caricamento: {e}")


Inizializzazione della struttura di Virchow2 con mlp_ratio corretto...
Caricamento e iniezione dei pesi locali da: /kaggle/input/datasets/irenebartolini/virchow2/uni_backbone_weights.bin

--- [SUCCESSO] Virchow2 è caricato e pronto al 100% in modalità offline! ---


In [12]:
!pip install -q monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.2 MB/s eta 0:00:00a 0:00:01


In [18]:
# import torch
# import torch.nn as nn
# from monai.networks.nets import UNETR

# class Virchow2UNETR(nn.Module):
#     def __init__(self, virchow_backbone, img_size=(512, 512), out_channels=1):
#         super().__init__()
#         # 1. Agganciamo il backbone offline di Virchow2
#         self.virchow = virchow_backbone
        
#         # Congeliamo i pesi di Virchow2
#         for param in self.virchow.parameters():
#             param.requires_grad = False
            
#         # 2. Inizializziamo il modulo UNETR di MONAI
#         self.unetr = UNETR(
#             in_channels=3,
#             out_channels=out_channels,     # 1 per la maschera del surrene
#             img_size=img_size,              # (512, 512)
#             feature_size=16,
#             hidden_size=1280,              # Dimensione embedding Virchow2
#             mlp_dim=6832,                  
#             num_heads=16,
#             proj_type="perceptron",
#             spatial_dims=2                 
#         )
        
#     def forward(self, x):
#         # x input ha dimensioni: [Batch_Size, 3, 512, 512]
#         B = x.shape[0]
        
#         # --- FASE 1: ESTRAZIONE FEATURE DA VIRCHOW2 ---
#         x_resized = nn.functional.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        
#         with torch.no_grad():
#             features = self.virchow.forward_features(x_resized) 
#             # Dimensione features: [B, 261, 1280]
            
#         # Rimuoviamo i 5 token speciali e isoliamo i 256 token spaziali
#         spatial_tokens = features[:, 5:, :] # Dimensione: [B, 256, 1280]
        
#         # --- FASE 2: PREPARAZIONE PER IL DECODER DI MONAI ---
#         # UNETR si aspetta che i token sequenziali vengano rimodellati in una griglia 2D (16x16)
#         # e che i canali siano nell'ordine corretto [B, Canali, Altezza, Larghezza]
#         hidden_states = spatial_tokens.transpose(1, 2).view(B, 1280, 16, 16)
        
#         # --- FASE 3: PASSAGGIO NEI BLOCCHI DEL DECODER DI MONAI ---
#         # Estraiamo le skip connections proiettando l'immagine di input e gli stati intermedi
#         enc0 = self.unetr.encoder1(x)                                  # Risoluzione 512x512
#         enc1 = self.unetr.encoder2(hidden_states)                      # Risoluzione 128x128
#         enc2 = self.unetr.encoder3(hidden_states)                      # Risoluzione 64x64
#         enc3 = self.unetr.encoder4(hidden_states)                      # Risoluzione 32x32
        
#         # Il collo di bottiglia profondo (bottleneck) a risoluzione 16x16
#         dec4 = self.unetr.encoder5(hidden_states)                      
        
#         # Upsampling progressivo tramite il decoder convoluzionale di MONAI
#         dec3 = self.unetr.decoder5(dec4, enc3)                         # Da 16x16 a 32x32
#         dec2 = self.unetr.decoder4(dec3, enc2)                         # Da 32x32 a 64x64
#         dec1 = self.unetr.decoder3(dec2, enc1)                         # Da 64x64 a 128x128
#         out = self.unetr.decoder2(dec1, enc0)                          # Da 128x128 a 512x512
        
#         # Mappa di classificazione finale (convoluzione 1x1 per sputare fuori out_channels)
#         logits = self.unetr.out(out)
        
#         return logits

In [19]:
# # 1. Istanziamo il modello ibrido passando il 'backbone' configurato nei passaggi precedenti
# modello_completo = Virchow2UNETR(virchow_backbone=backbone, img_size=(512, 512), out_channels=1)
# modello_completo.to("cuda" if torch.cuda.is_available() else "cpu")
# modello_completo.eval()

# # 2. Creiamo un finto batch di dati (es. 2 patch istologici di dimensione 512x512)
# finto_input_slide = torch.randn(2, 3, 512, 512).to("cuda" if torch.cuda.is_available() else "cpu")

# # 3. Facciamo il pass in avanti (Forward)
# print("Esecuzione del forward test...")
# with torch.no_grad():
#     maschera_surrene_predetta = modello_completo(finto_input_slide)

# print(f"Dimensione dell'input originale: {finto_input_slide.shape}")
# print(f"Dimensione della maschera predetta: {maschera_surrene_predetta.shape}")

# # Verifica finale
# assert maschera_surrene_predetta.shape == (2, 1, 512, 512), "Errore di mismatch geometrico!"
# print("\n--- [SUCCESSO] La UNet riceve correttamente le feature di Virchow2 e genera la maschera 512x512! ---")

Esecuzione del forward test...


AttributeError: 'UNETR' object has no attribute 'encoder5'